In [0]:
# Multiple ways to read a Data

'''
1. Postgres & MySQL using JDBC 
2. S3
3. Snowflake
4. DBFS 
5. ADLS
6. RDS
'''

In [0]:
# -----------------------------------------------
# Read data from PostgreSQL database using JDBC
# -----------------------------------------------


# Modern way: keep only connection details in URL
# credentials will be passed separately using properties
jdbc_url = "jdbc:postgresql://razz-db-therazz007-878c.b.aivencloud.com:22917/defaultdb"


# JDBC connection properties
# driver  -> JDBC driver required for PostgreSQL
# user    -> database username
# password-> database password
properties = {
     "driver": "org.postgresql.Driver",   
    "user": "avnadmin",
    "password": "<DBPASSWORD>"
}


# Read data from PostgreSQL table "orders"
# Spark internally runs: SELECT * FROM orders
# and converts the result into a Spark DataFrame
df = spark.read.jdbc(jdbc_url, table="orders", properties=properties)


# Display the first 20 rows of the dataframe
df.show()

In [0]:
# -----------------------------------------------
# Read data using format("jdbc") and option()
# -----------------------------------------------

df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://razz-db-therazz007-878c.b.aivencloud.com:22917/defaultdb") \
    .option("dbtable", "orders") \
    .option("user", "avnadmin") \
    .option("password", "<DBPASSWORD>") \
    .option("driver", "org.postgresql.Driver") \
    .load()


# Show data
df.show()

In [0]:
# Example: Reading a public Parquet file from the NYC Taxi dataset
path = "s3a://nyc-tlc/trip data/yellow_tripdata_2023-01.parquet"

df = spark.read.parquet(path)
df.show(5)

In [0]:
# Run SQL query on Delta table
df = spark.sql("""
SELECT city_name, country_code, cloud_base_height
FROM samples.accuweather.forecast_hourly_imperial
LIMIT 10
""")

# Show result
df.show()

## Read CSV Data from DBFS Volume



In [0]:
# how to read csv data from DBFS volume 


path = r'/Volumes/workspace/default/products-internal/products-bulk(Sheet1) (2).csv'


df = spark.read.csv(path,header=True,inferSchema=True)
df.show(5,truncate=False)



In [0]:

df = spark.read.csv("dbfs:/Volumes/workspace/default/products-internal/products-bulk(Sheet1) (2).csv",header=True,inferSchema=True)
df.show(5,truncate=False)


In [0]:
# another ways to read csv 


options = {
    "header": "true", 
    "inferSchema": "true",
    "delimiter": ",",
    "quote": "\"",
    "escape": "\"",
    "multiLine": "true",

}
df  = spark.read.format("csv").options(**options).option("path","dbfs:/Volumes/workspace/default/products-internal/products-bulk(Sheet1) (2).csv").load()
df.show(5,truncate=False)

## Read JSON Data


In [0]:
jsonPath = "/Volumes/workspace/default/products-internal/products.json"


properties = {
    "multiline": "true",
 
    "inferSchema": "true",
    "dateFormat": "yyyy-MM-dd'T'HH:mm:ss",
    "timestampFormat": "yyyy-MM-dd'T'HH:mm:ss",
    "locale": "en-US",
    "encoding": "UTF-8",
}

df = spark.read.format("json").options(**properties).option("path",jsonPath).load()
df.show(5,truncate=False)



df.printSchema()


In [0]:
jsonPath = "/Volumes/workspace/default/products-internal/products.json"

df = spark.read \
    .option("multiline", "true") \
    .json(jsonPath)

df.show() 

In [0]:
from pyspark.sql.functions import explode

df_products = df.select(
    explode("products.data.items").alias("product")
)
df_products.show(1)

df_final = df_products.select("product.*")

df_final.show(1)